In [ ]:
# Install required packages - run this first
!pip install google-generativeai
!pip install elevenlabs
!pip install langchain
!pip install langchain-core
!pip install langchain-community
!pip install langchain-google-genai
!pip install gradio
!pip install requests

In [ ]:
# Basic imports
import os
import re
import requests
import json
import gradio as gr

# Google Gemini imports
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate


# legacy chains + memory now live in langchain_classic
from langchain_classic.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory


# ElevenLabs imports
from elevenlabs.client import ElevenLabs
from elevenlabs import save

### API Setup

Before running this notebook, you need to get API keys:

**Google Gemini API Key:**
1. Go to https://aistudio.google.com/app/api-keys
2. Click "Create API Key"
3. Copy your key and paste it below

**ElevenLabs API Key:**
1. Sign up at https://elevenlabs.io/
2. Go to "Developer" section
3. Copy your API key from the "API Key" section

In [ ]:
# Google Gemini API Key
GEMINI_API_KEY = "..."

# ElevenLabs API Key
ELEVENLABS_API_KEY = "..."

# ElevenLabs Voice ID (Rachel voice by default)
ELEVENLABS_VOICE_ID = "21m00Tcm4TlvDq8ikWAM"

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Initialize ElevenLabs client
elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

print("✅ API keys configured successfully!")


In [ ]:
template = """You are a helpful assistant to answer user queries.
{chat_history}
User: {user_message}
Chatbot:"""

prompt = PromptTemplate(
    input_variables=["chat_history", "user_message"],
    template=template
)

memory = ConversationBufferMemory(memory_key="chat_history")

print("✅ Prompt template created!")

In [ ]:
# Initialize Gemini model using direct Google GenerativeAI (NOT LangChain wrapper)
import google.generativeai as genai

# Configure the Gemini model directly
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

# Create a custom LLM wrapper for LangChain compatibility
class GeminiLLM:
    def __init__(self, model):
        self.model = model
        self.memory_history = []

    def predict(self, user_message):
        # Build conversation context
        full_prompt = f"You are a helpful assistant to answer user queries.\n"
        for msg in self.memory_history:
            full_prompt += f"{msg}\n"
        full_prompt += f"User: {user_message}\nChatbot:"

        # Generate response
        response = self.model.generate_content(full_prompt)
        answer = response.text

        # Update memory
        self.memory_history.append(f"User: {user_message}")
        self.memory_history.append(f"Chatbot: {answer}")

        # Keep only last 10 exchanges to avoid token limits
        if len(self.memory_history) > 20:
            self.memory_history = self.memory_history[-20:]

        return answer

# Initialize the custom LLM
llm_chain = GeminiLLM(gemini_model)

print("✅ Gemini LLM initialized with direct SDK!")


In [ ]:
def generate_audio_elevenlabs(text):
    """
    Generate audio using ElevenLabs API (Updated method)
    Returns audio file path or error message
    """
    try:
        # Generate audio using the newer SDK method
        audio = elevenlabs_client.text_to_speech.convert(
            voice_id=ELEVENLABS_VOICE_ID,
            text=text,
            model_id="eleven_multilingual_v2",
            output_format="mp3_44100_128"
        )

        # Save audio to file
        output_path = f"/content/output_audio_{hash(text) % 10000}.mp3"

        # Write the audio bytes to file
        with open(output_path, 'wb') as f:
            for chunk in audio:
                f.write(chunk)

        print(f"✅ Audio saved to: {output_path}")  # Debug print

        return {
            "type": "SUCCESS",
            "response": output_path,
            "message": "Audio generated successfully"
        }
    except Exception as e:
        print(f"❌ ElevenLabs Error: {str(e)}")  # Debug print
        return {
            "type": "ERROR",
            "response": str(e),
            "message": f"Audio generation failed: {str(e)}"
        }


In [ ]:
def get_audio_reply_for_question(text):
    """
    Generate audio for the chatbot response
    """
    generated_audio_event = generate_audio_elevenlabs(text)

    final_response = {
        "audio_path": '',
        "message": ''
    }

    if generated_audio_event["type"] == "SUCCESS":
        audio_path = generated_audio_event["response"]
        final_response['audio_path'] = audio_path
        final_response['message'] = "Audio generated successfully"
    else:
        final_response['message'] = generated_audio_event['message']

    return final_response

print("✅ Audio reply function defined!")


In [ ]:
def get_text_response(user_message):
    """
    Get text response from Gemini
    """
    try:
        response = llm_chain.predict(user_message=user_message)
        return response
    except Exception as e:
        error_msg = f"Error in Gemini response: {str(e)}"
        print(error_msg)
        return f"Sorry, I encountered an error: {str(e)}"

print("✅ Text response function defined!")


In [ ]:
def get_text_response_and_audio_response(user_message):
    """
    Get both text response from Gemini and audio from ElevenLabs
    """
    # Get text response from Gemini
    text_response = get_text_response(user_message)

    # Generate audio for the response
    audio_reply = get_audio_reply_for_question(text_response)

    final_response = {
        'text': text_response,
        'audio_path': audio_reply.get('audio_path', ''),
        'message': audio_reply.get('message', '')
    }

    return final_response

print("✅ Combined response function defined!")


In [ ]:
def chat_bot_response(message, history):
    """
    Main chatbot function for Gradio interface
    Returns tuple of (text_response, audio_file_path)
    """
    try:
        # Get text and audio response
        response = get_text_response_and_audio_response(message)

        text_response = response['text']
        audio_path = response['audio_path']

        if audio_path and os.path.exists(audio_path):
            # Return both text and audio
            return (text_response, audio_path)  # ✅ Return tuple!
        else:
            # Return only text if audio fails
            return (text_response, None)  # ✅ Return tuple with None

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(error_msg)
        return (error_msg, None)


In [ ]:
# Create a custom Gradio interface with audio support
def chat_with_audio(message, history):
    """Modified chatbot function that returns message tuple and audio"""
    try:
        response = get_text_response_and_audio_response(message)
        text_response = response['text']
        audio_path = response['audio_path'] if response['audio_path'] else None

        # Return the bot response and audio path
        return text_response, audio_path
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(error_msg)
        return error_msg, None

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Gemini + ElevenLabs Chatbot")
    gr.Markdown("Chat with Google Gemini AI with voice responses from ElevenLabs")

    with gr.Row():
        with gr.Column():
            chatbot = gr.Chatbot(height=400, label="Chat History")
            msg = gr.Textbox(
                label="Your Message",
                placeholder="Type your message here...",
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear Chat")

            audio_output = gr.Audio(
                label="🔊 Voice Response",
                autoplay=True,
                visible=True
            )

    # Example questions
    gr.Examples(
        examples=[
            "How are you doing?",
            "What are your interests?",
            "Tell me a short story",
            "What's the weather like today?",
            "Explain quantum computing in simple terms"
        ],
        inputs=msg
    )

    def respond(message, chat_history):
        if not message.strip():
            return "", chat_history, None

        # Get response
        bot_message, audio_path = chat_with_audio(message, chat_history)

        # Update chat history
        chat_history.append((message, bot_message))

        return "", chat_history, audio_path

    # Connect events
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot, audio_output])
    msg.submit(respond, [msg, chatbot], [msg, chatbot, audio_output])
    clear_btn.click(lambda: (None, None), None, [chatbot, audio_output])

print("✅ Gradio interface with audio created!")

In [ ]:
if __name__ == "__main__":
    # Launch with public link
    demo.launch(
        share=True,  # Creates public link
        debug=True   # Shows errors and logs
    )


### Publish your code with below steps

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

In [ ]:
HUGGING_FACE_REPO_ID = "<<Hugging Face UserName/Repo ID>>"

**Adding Secret Variables in Hugging Face Account:**

- Open your Space
- Click on Settings Button
- Checkout to **Variables and secrets** section
- Create New Secrets

In [ ]:
%mkdir /content/ChatBotWithGeminiLangChainAndElevenLabs
!wget -P  /content/ChatBotWithGeminiLangChainAndElevenLabs/ https://s3.ap-south-1.amazonaws.com/cdn1.ccbp.in/GenAI-Workshop/ChatBotWithGeminiLangChainElevenLabs/app.py
!wget -P /content/ChatBotWithGeminiLangChainAndElevenLabs/ https://s3.ap-south-1.amazonaws.com/cdn1.ccbp.in/GenAI-Workshop/ChatBotWithGeminiLangChainElevenLabs/requirements.txt

In [ ]:
%cd /content/ChatBotWithGeminiLangChainAndElevenLabs
api.upload_file(
    path_or_fileobj="./requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=HUGGING_FACE_REPO_ID,
    repo_type="space")

api.upload_file(
    path_or_fileobj="./app.py",
    path_in_repo="app.py",
    repo_id=HUGGING_FACE_REPO_ID,
    repo_type="space")